## init

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

from collections import defaultdict

BATCH_SIZE = 128
MAX_CONTEXT_LENGTH = 10
PAD_ITEM_ID = 0

# Ids

## books

In [2]:
books = pd.read_csv('data/processed/books_enriched.csv', low_memory=False, delimiter=';')
books.info()

<class 'pandas.DataFrame'>
RangeIndex: 14502 entries, 0 to 14501
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   kode_buku                 14502 non-null  str    
 1   isbn_issn                 12794 non-null  str    
 2   judul_buku                14502 non-null  str    
 3   subjudul                  7818 non-null   str    
 4   penulis                   13490 non-null  str    
 5   nama_belakang             13838 non-null  str    
 6   topik                     13806 non-null  str    
 7   deskripsi                 14502 non-null  str    
 8   bahasa                    14502 non-null  str    
 9   kategori                  14502 non-null  str    
 10  tahun_terbit              14305 non-null  float64
 11  key                       14502 non-null  str    
 12  confidence                14257 non-null  float64
 13  volume_info               13681 non-null  str    
 14  isbn_10_enrich   

In [3]:
books[['book_id','judul_full']].head(5)

,book_id,judul_full
0,B000003,Preparing for the Twenty-first Century
1,B000005,The Illustrator 9 Wow! Book
2,B000006,Crucial Decisions leadership in policymaking a...
3,B000010,Basic Biochemistry
4,B000017,Retail Management a strategic approach


In [4]:
# make bookId
books['bookId'] = books.index + 1
books[['book_id', 'bookId','judul_full']].head(10)

,book_id,bookId,judul_full
0,B000003,1,Preparing for the Twenty-first Century
1,B000005,2,The Illustrator 9 Wow! Book
2,B000006,3,Crucial Decisions leadership in policymaking a...
3,B000010,4,Basic Biochemistry
4,B000017,5,Retail Management a strategic approach
5,B000019,6,The Capital Budgeting Decision economic analys...
6,B000022,7,Logistical Management a systems integration of...
7,B000025,8,Heat Transfer
8,B000027,9,Principles of Horticulture
9,B000044,10,Elements of Econometrics


In [5]:
id_to_books = dict(zip(books['bookId'], books['book_id']))
id_to_books[0] = ""

id_to_books

{1: 'B000003',
 2: 'B000005',
 3: 'B000006',
 4: 'B000010',
 5: 'B000017',
 6: 'B000019',
 7: 'B000022',
 8: 'B000025',
 9: 'B000027',
 10: 'B000044',
 11: 'B000045',
 12: 'B000051',
 13: 'B000052',
 14: 'B000057',
 15: 'B000058',
 16: 'B000060',
 17: 'B000061',
 18: 'B000063',
 19: 'B000073',
 20: 'B000079',
 21: 'B000080',
 22: 'B000086',
 23: 'B000087',
 24: 'B000088',
 25: 'B000089',
 26: 'B000090',
 27: 'B000091',
 28: 'B000092',
 29: 'B000094',
 30: 'B000095',
 31: 'B000096',
 32: 'B000097',
 33: 'B000098',
 34: 'B000099',
 35: 'B000100',
 36: 'B000101',
 37: 'B000102',
 38: 'B000113',
 39: 'B000118',
 40: 'B000120',
 41: 'B000123',
 42: 'B000129',
 43: 'B000130',
 44: 'B000134',
 45: 'B000135',
 46: 'B000136',
 47: 'B000137',
 48: 'B000139',
 49: 'B000141',
 50: 'B000144',
 51: 'B000145',
 52: 'B000151',
 53: 'B000154',
 54: 'B000158',
 55: 'B000159',
 56: 'B000161',
 57: 'B000167',
 58: 'B000174',
 59: 'B000179',
 60: 'B000180',
 61: 'B000182',
 62: 'B000185',
 63: 'B000186',
 

In [6]:
books_to_id = dict(zip(books['book_id'], books['bookId']))
books_to_id[''] = 0
books_to_id

{'B000003': 1,
 'B000005': 2,
 'B000006': 3,
 'B000010': 4,
 'B000017': 5,
 'B000019': 6,
 'B000022': 7,
 'B000025': 8,
 'B000027': 9,
 'B000044': 10,
 'B000045': 11,
 'B000051': 12,
 'B000052': 13,
 'B000057': 14,
 'B000058': 15,
 'B000060': 16,
 'B000061': 17,
 'B000063': 18,
 'B000073': 19,
 'B000079': 20,
 'B000080': 21,
 'B000086': 22,
 'B000087': 23,
 'B000088': 24,
 'B000089': 25,
 'B000090': 26,
 'B000091': 27,
 'B000092': 28,
 'B000094': 29,
 'B000095': 30,
 'B000096': 31,
 'B000097': 32,
 'B000098': 33,
 'B000099': 34,
 'B000100': 35,
 'B000101': 36,
 'B000102': 37,
 'B000113': 38,
 'B000118': 39,
 'B000120': 40,
 'B000123': 41,
 'B000129': 42,
 'B000130': 43,
 'B000134': 44,
 'B000135': 45,
 'B000136': 46,
 'B000137': 47,
 'B000139': 48,
 'B000141': 49,
 'B000144': 50,
 'B000145': 51,
 'B000151': 52,
 'B000154': 53,
 'B000158': 54,
 'B000159': 55,
 'B000161': 56,
 'B000167': 57,
 'B000174': 58,
 'B000179': 59,
 'B000180': 60,
 'B000182': 61,
 'B000185': 62,
 'B000186': 63,
 

## user

In [7]:
users = pd.read_csv('data/users/all_users.csv')
users.info()

<class 'pandas.DataFrame'>
RangeIndex: 1798 entries, 0 to 1797
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Unnamed: 0.1  1798 non-null   int64  
 1   Unnamed: 0    1798 non-null   int64  
 2   ID Anggota    1798 non-null   int64  
 3   prodi_id      1785 non-null   str    
 4   jurusan       1785 non-null   str    
 5   fakultas_id   1786 non-null   float64
 6   fakultas      1785 non-null   str    
 7   jenjang       1785 non-null   str    
 8   Nama Anggota  1798 non-null   str    
 9   role          1798 non-null   str    
 10  angkatan      1775 non-null   float64
 11  userId        1798 non-null   int64  
dtypes: float64(2), int64(4), str(6)
memory usage: 306.8 KB


In [8]:
users['userId'] = users.index + 1
users.head(10)

,Unnamed: 0.1,Unnamed: 0,ID Anggota,prodi_id,jurusan,fakultas_id,fakultas,jenjang,Nama Anggota,role,angkatan,userId
0,0,0,196207282021211000,12.0,Manajemen,1.0,Fakultas Ekonomi dan Bisnis,S1,Bowo Santoso,lecturer,NaN,1
1,1,1,196211201991032000,65.0,Magister Ilmu Lingkungan,3.0,Fakultas Teknik dan Sains,S2,"Dr.Ir.Susilowati,Mt",lecturer,NaN,2
2,2,2,196509291992032000,13.0,Akuntansi,1.0,Fakultas Ekonomi dan Bisnis,S1,Dra Ec Sri Trisnaningsih Msi,lecturer,NaN,3
3,3,3,196701231993032000,13.0,Akuntansi,1.0,Fakultas Ekonomi dan Bisnis,S1,Dra Ec Tituk Diah Widayantie M Aks,lecturer,NaN,4
4,4,4,197708172021212000,53.0,Desain Interior,5.0,Fakultas Arsitektur dan Desain,S1,Dyan Agustin,lecturer,NaN,5
5,5,5,198511062019031000,52.0,Desain Komunikasi Visual,5.0,Fakultas Arsitektur dan Desain,S1,Aris Sutejo,lecturer,NaN,6
6,6,6,198706172025062000,11.0,Ekonomi Pembangunan,1.0,Fakultas Ekonomi dan Bisnis,S1,"Dr. Eunike Rose Mita Lukiani, M.Pd",lecturer,NaN,7
7,7,7,199010282024062000,37.0,Fisika,3.0,Fakultas Teknik dan Sains,S1,Nenni Mona Aruan,lecturer,NaN,8
8,8,8,199312042024061000,37.0,Fisika,3.0,Fakultas Teknik dan Sains,S1,Fajar Timur,lecturer,NaN,9
9,9,9,199509302024062000,24.0,Agribisnis,2.0,Fakultas Pertanian,S1,Yesi Mustika Ningsih,lecturer,NaN,10


In [9]:
id_to_users = dict(zip(users['userId'], users['ID Anggota']))
id_to_users[0] = ""
id_to_users

{1: 196207282021211000,
 2: 196211201991032000,
 3: 196509291992032000,
 4: 196701231993032000,
 5: 197708172021212000,
 6: 198511062019031000,
 7: 198706172025062000,
 8: 199010282024062000,
 9: 199312042024061000,
 10: 199509302024062000,
 11: 199901062024062000,
 12: 17041010008,
 13: 18011010001,
 14: 18011010018,
 15: 18025010010,
 16: 18033010059,
 17: 18041010032,
 18: 19011010107,
 19: 19012010076,
 20: 19012010248,
 21: 19013010276,
 22: 19024010150,
 23: 19025010017,
 24: 19025010021,
 25: 19025010150,
 26: 19031010119,
 27: 19032010094,
 28: 19041010041,
 29: 19041010204,
 30: 19043010024,
 31: 19043010076,
 32: 19043010135,
 33: 19043010156,
 34: 19043010194,
 35: 19081010020,
 36: 19082010024,
 37: 19900308019,
 38: 20011010001,
 39: 20011010002,
 40: 20011010006,
 41: 20011010010,
 42: 20011010034,
 43: 20011010038,
 44: 20011010047,
 45: 20011010057,
 46: 20011010059,
 47: 20011010066,
 48: 20011010067,
 49: 20011010069,
 50: 20011010088,
 51: 20011010089,
 52: 200110101

In [10]:
users_to_id = dict(zip(users['ID Anggota'], users['userId']))
users_to_id[''] = 0
users_to_id

{196207282021211000: 1,
 196211201991032000: 2,
 196509291992032000: 3,
 196701231993032000: 4,
 197708172021212000: 5,
 198511062019031000: 6,
 198706172025062000: 7,
 199010282024062000: 8,
 199312042024061000: 9,
 199509302024062000: 10,
 199901062024062000: 11,
 17041010008: 12,
 18011010001: 13,
 18011010018: 14,
 18025010010: 15,
 18033010059: 16,
 18041010032: 17,
 19011010107: 18,
 19012010076: 19,
 19012010248: 20,
 19013010276: 21,
 19024010150: 22,
 19025010017: 23,
 19025010021: 24,
 19025010150: 25,
 19031010119: 26,
 19032010094: 27,
 19041010041: 28,
 19041010204: 29,
 19043010024: 30,
 19043010076: 31,
 19043010135: 32,
 19043010156: 33,
 19043010194: 34,
 19081010020: 35,
 19082010024: 36,
 19900308019: 37,
 20011010001: 38,
 20011010002: 39,
 20011010006: 40,
 20011010010: 41,
 20011010034: 42,
 20011010038: 43,
 20011010047: 44,
 20011010057: 45,
 20011010059: 46,
 20011010066: 47,
 20011010067: 48,
 20011010069: 49,
 20011010088: 50,
 20011010089: 51,
 20011010100: 

In [11]:
users[users['ID Anggota'] == 22082010145]

,Unnamed: 0.1,Unnamed: 0,ID Anggota,prodi_id,jurusan,fakultas_id,fakultas,jenjang,Nama Anggota,role,angkatan,userId
676,676,676,22082010145,Sistem Informasi,8.0,82.0,Fakultas Ilmu Komputer,S1,Mafda Khoirotul Fatha,student,2022.0,677


In [12]:
users.to_csv('data/users/all_users.csv')

# trans

In [13]:
trans_pd = pd.read_csv('data/processed/transactions_enriched.csv', low_memory=False, delimiter=';')
trans_pd.info()

<class 'pandas.DataFrame'>
RangeIndex: 5372 entries, 0 to 5371
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   ID Anggota          5372 non-null   int64
 1   Nama Anggota        5372 non-null   str  
 2   Kode Eksemplar      5372 non-null   int64
 3   Judul               5372 non-null   str  
 4   Tanggal Pinjam      5372 non-null   str  
 5   tahun_pinjam        5372 non-null   int64
 6   bulan_pinjam        5372 non-null   int64
 7   bulan_tahun_pinjam  5372 non-null   str  
 8   book_id             5372 non-null   str  
dtypes: int64(4), str(5)
memory usage: 830.7 KB


In [14]:
trans_pd.sample(5)

,ID Anggota,Nama Anggota,Kode Eksemplar,Judul,Tanggal Pinjam,tahun_pinjam,bulan_pinjam,bulan_tahun_pinjam,book_id
1712,18011010001,Winarti,91115303,Pengelolaan Keuangan Pada Satuan Kerja Perangk...,2024-05-14,2024,5,2024-05,B018260
806,21011010021,ALYA ROHANI,90888602,Manajemen Perbankan : Teori dan Aplikasi,2023-12-18,2023,12,2023-12,B023412
314,23011010087,Anggita Eka Prameswari,90922503,Etika bisnis : konsep dan kasus,2023-10-17,2023,10,2023-10,B024880
1028,197708172021212000,Dyan Agustin,90840703,Arsitektur dan perilaku manusia,2024-01-12,2024,1,2024-01,B024891
1810,22012010025,LINTANG KINANTHI KUSUMAWARDANI,91101605,"Perilaku dan manajemen organisasi , Jilid 2",2024-05-31,2024,5,2024-05,B017987


In [15]:
trans_pd['userId'] = trans_pd['ID Anggota'].map(users_to_id)
trans_pd['bookId'] = trans_pd['book_id'].map(books_to_id)

trans_pd.sample(5)

,ID Anggota,Nama Anggota,Kode Eksemplar,Judul,Tanggal Pinjam,tahun_pinjam,bulan_pinjam,bulan_tahun_pinjam,book_id,userId,bookId
1156,21035010005,MUHAMMAD AINUN NASRUL MAJID,91075001,Modul ajar teknik gempa,2024-02-15,2024,2,2024-02,B018446,329,12263
1130,22013010238,TALITHA RAISSA ASMAWATI,91081803,Sistem informasi manajemen,2024-02-07,2024,2,2024-02,B019904,525,12695
1088,20013010224,Gisella Chantika Neyza,91297408,"Partial least squares : konsep, teknik dan apl...",2024-01-30,2024,1,2024-01,B024742,97,13923
1171,23032010161,Pinto Legowo Abhima Putra,90313603,HEAT TRANSFER,2024-02-16,2024,2,2024-02,B000231,869,79
939,22011010159,ANGGIE WAHYUWARDANI,923021705,Mikroekonomi : Teori pengantar,2023-12-20,2023,12,2023-12,B020279,454,12802


In [16]:
trans_pd['timestamp'] = pd.to_datetime(trans_pd['Tanggal Pinjam'])
trans_pd.info()

<class 'pandas.DataFrame'>
RangeIndex: 5372 entries, 0 to 5371
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   ID Anggota          5372 non-null   int64         
 1   Nama Anggota        5372 non-null   str           
 2   Kode Eksemplar      5372 non-null   int64         
 3   Judul               5372 non-null   str           
 4   Tanggal Pinjam      5372 non-null   str           
 5   tahun_pinjam        5372 non-null   int64         
 6   bulan_pinjam        5372 non-null   int64         
 7   bulan_tahun_pinjam  5372 non-null   str           
 8   book_id             5372 non-null   str           
 9   userId              5372 non-null   int64         
 10  bookId              5372 non-null   int64         
 11  timestamp           5372 non-null   datetime64[us]
dtypes: datetime64[us](1), int64(6), str(5)
memory usage: 956.6 KB


In [17]:
trans_pd['timestamp'].sample()

1746   2024-05-17
Name: timestamp, dtype: datetime64[us]

In [18]:
trans_pd.isna().sum()

ID Anggota            0
Nama Anggota          0
Kode Eksemplar        0
Judul                 0
Tanggal Pinjam        0
tahun_pinjam          0
bulan_pinjam          0
bulan_tahun_pinjam    0
book_id               0
userId                0
bookId                0
timestamp             0
dtype: int64

In [19]:
trans = trans_pd[['userId', 'bookId', 'timestamp']]

# Sequence

In [20]:
def build_sequence(transaction_data):
    sequences = defaultdict(list)
    for userId, bookId, timestamp in trans.values:
            sequences[userId].append(
                {
                    "bookId": bookId,
                    "timestamp": timestamp,
                }
            )
    # Sort sequences by timestamp for every user.
    for user_id, context in sequences.items():
        context.sort(key=lambda x: x["timestamp"])
        sequences[user_id] = context

    return sequences

In [21]:
sequences = build_sequence(trans)

## Negative Sequence

In [22]:
books_count = books.shape[0]
books_count

14502

In [23]:
def build_negative_seq(sequences):
    examples = {
        "sequence": [],
        "negative_sequence": [],
    }

    for userId in sequences:
        sequence = [int(d["bookId"]) for d in sequences[userId]]

        # Get negative sequence.
        def random_negative_item_id(low, high, positive_lst):
            sampled = np.random.randint(low=low, high=high)
            while sampled in positive_lst:
                sampled = np.random.randint(low=low, high=high)
            return sampled

        negative_sequence = [
            random_negative_item_id(1, books_count + 1, sequence)
            for _ in range(len(sequence))
        ]

        examples["sequence"].append(np.array(sequence))
        examples["negative_sequence"].append(np.array(negative_sequence))

    examples["sequence"] = tf.ragged.constant(examples["sequence"])
    examples["negative_sequence"] = tf.ragged.constant(examples["negative_sequence"])

    return examples

In [24]:
examples = build_negative_seq(sequences)

## Split Data Train Val Test

In [25]:
from collections import defaultdict

def data_split(sequences):
    user_train, user_valid, user_test = {}, {}, {}
    itemnum = 0

    for user, seq in sequences.items():
        # sort by timestamp
        seq_sorted = sorted(seq, key=lambda x: x["timestamp"])
        book_ids = [int(x["bookId"]) for x in seq_sorted]
        itemnum = max(itemnum, max(book_ids))

        n = len(book_ids)

        if n < 2:
            # Panjang 1: Tidak bisa diuji
            user_train[user] = book_ids
            user_valid[user] = []
            user_test[user] = []
        elif n == 2:
            # Panjang 2:
            user_train[user] = book_ids
            user_valid[user] = []
            user_test[user] = []
        elif n == 3:
            # Panjang 3:
            user_train[user] = book_ids[:-1] 
            user_valid[user] = []
            user_test[user] = book_ids
        else:
            # Panjang > 3
            user_train[user] = book_ids[:-2]
            user_valid[user] = book_ids[:-1]
            user_test[user] = book_ids

    return user_train, user_valid, user_test, itemnum

In [26]:
train, valid, test, itemnum = data_split(sequences)

# Padding

In [27]:
def build_tf_dataset(user_dict, itemnum):
    sequences = []
    negative_sequences = []

    for u, seq in user_dict.items():
        if not seq: continue
        # Ensure sequence is a NumPy array for consistency
        seq_np = np.array(seq, dtype=np.int32)
        # Generate negative samples for the sequence length
        negative_seq_np = np.random.randint(1, itemnum + 1, size=len(seq_np))

        sequences.append(seq_np)
        negative_sequences.append(negative_seq_np)

    # Convert lists of NumPy arrays (sequences) into Ragged Tensors
    ragged_examples = {
        "sequence": tf.ragged.constant(sequences, dtype=tf.int32),
        "negative_sequence": tf.ragged.constant(negative_sequences, dtype=tf.int32),
    }

    # Create the dataset from the ragged tensors, and then batch it
    ds = tf.data.Dataset.from_tensor_slices(ragged_examples)

    return ds

In [28]:
def _preprocess(example, train=True):
    seq = example["sequence"]
    neg_seq = example["negative_sequence"]
    batch_size = tf.shape(seq)[0]

    if train:
        # Example seq: [A, B, C] -> input: [A, B], target: [B, C]
        inputs = seq[..., :-1]
        targets = seq[..., 1:]
        neg_targets = neg_seq[..., 1:]

        # Truncate to MAX_CONTEXT_LENGTH
        inputs = inputs[..., -MAX_CONTEXT_LENGTH:]
        targets = targets[..., -MAX_CONTEXT_LENGTH:]
        neg_targets = neg_targets[..., -MAX_CONTEXT_LENGTH:]

        # Pad dynamically
        inputs_padded = inputs.to_tensor(shape=[batch_size, MAX_CONTEXT_LENGTH], default_value=PAD_ITEM_ID)
        targets_padded = targets.to_tensor(shape=[batch_size, MAX_CONTEXT_LENGTH], default_value=PAD_ITEM_ID)
        neg_targets_padded = neg_targets.to_tensor(shape=[batch_size, MAX_CONTEXT_LENGTH], default_value=PAD_ITEM_ID)

        # Sample weight: 1.0 for real target items, 0.0 for padding
        sample_weight = tf.cast(targets_padded != PAD_ITEM_ID, dtype="float32")

        return (
            {
                "item_ids": inputs_padded,
                "padding_mask": inputs_padded != PAD_ITEM_ID,
            },
            {
                "positive_sequence": targets_padded,
                "negative_sequence": neg_targets_padded,
            },
            sample_weight
        )
    else:
        # For testing: Context is everything EXCEPT the last item. Target IS the last item.
        # Example seq: [A, B, C] -> context: [A, B], target: [C]
        context = seq[..., :-1]
        target = seq[..., -1:]

        # Truncate context
        context = context[..., -MAX_CONTEXT_LENGTH:]
        
        context_padded = context.to_tensor(shape=[batch_size, MAX_CONTEXT_LENGTH], default_value=PAD_ITEM_ID)
        
        # Flatten target into a 1D tensor [batch_size]
        target_flat = tf.reshape(target.to_tensor(), [batch_size])

        return (
            {
                "item_ids": context_padded,
                "padding_mask": context_padded != PAD_ITEM_ID,
            },
            {
                "positive_sequence": target_flat,
            }
        )

In [29]:
def preprocess_train(examples):
    return _preprocess(examples, train=True)
def preprocess_val(examples):
    return _preprocess(examples, train=False)
def preprocess_test(examples):
    return _preprocess(examples, train=False)

In [30]:
user_train, user_valid, user_test, itemnum = data_split(sequences)

In [31]:
print(f"Jumlah user di Train set: {sum(1 for v in user_train.values() if v)}")
print(f"Jumlah user di Valid set: {sum(1 for v in user_valid.values() if v)}")
print(f"Jumlah user di Test set:  {sum(1 for v in user_test.values() if v)}")

Jumlah user di Train set: 1798
Jumlah user di Valid set: 396
Jumlah user di Test set:  633


In [32]:
train_ds = build_tf_dataset(user_train, itemnum).batch(BATCH_SIZE).map(preprocess_train)
val_ds = build_tf_dataset(user_valid, itemnum).batch(BATCH_SIZE).map(preprocess_val)
test_ds = build_tf_dataset(user_test, itemnum).batch(BATCH_SIZE).map(preprocess_test)

# cold warm set

## users

In [33]:
import math

# sequence_lengths = [len(seq) for seq in sequences.values()]
# print(f"max seq length: {max(sequence_lengths)}")
# percentile_20 = np.percentile(sequence_lengths, 20)
# user_threshold = math.ceil(percentile_20)

testable_seq_lengths = [len(seq) for seq in sequences.values() if len(seq) >= 2]
user_threshold = np.percentile(testable_seq_lengths, 20)

print(f"Ambang Batas (Threshold) Pengguna: {user_threshold} interaksi")

Ambang Batas (Threshold) Pengguna: 2.0 interaksi


In [34]:
cold_users = set()
warm_users = set()

for user, seq in sequences.items():
    if len(seq) <= user_threshold:
        cold_users.add(user)
    else:
        warm_users.add(user)

print(f"Total Cold Users: {len(cold_users)}")
print(f"Total Warm Users: {len(warm_users)}")

Total Cold Users: 1165
Total Warm Users: 633


## items

In [35]:
warm_items = set()

# Item yang muncul pada train set adalah warm-item
for user, seq in user_train.items():
    warm_items.update(seq)

# Mencari total seluruh item unik yang ada (berdasarkan dictionary sequence awal)
all_items_in_data = set()
for user, seq in sequences.items():
    for interaksi in seq:
        all_items_in_data.add(int(interaksi["bookId"]))

# Item yang ada di dataset tapi TIDAK ada di train set adalah cold-item
cold_items = all_items_in_data - warm_items

print(f"Total Warm Items: {len(warm_items)}")
print(f"Total Cold Items: {len(cold_items)}")

Total Warm Items: 1747
Total Cold Items: 213


In [37]:
books['type'] = 'cold'
books.loc[books['bookId'].isin(warm_items), 'type'] = 'warm'
books.to_csv('data/processed/books_enriched_with_type.csv', index=False, sep=';')

In [ ]:
books['type'].value_counts()

type
cold    12755
warm     1747
Name: count, dtype: int64

In [ ]:
books[books['judul'] == ]